In [1]:
import psycopg2
import json
from decimal import Decimal
from datetime import datetime, date, time
from uuid import UUID


class PostgresDataEncoder(json.JSONEncoder):
    """
    Custom JSON encoder to handle PostgreSQL native types.
    Converts datetime, date, time, UUID, Decimal into JSON-serializable types.
    """
    def default(self, obj):
        if isinstance(obj, (datetime, date, time)):
            return obj.isoformat()
        elif isinstance(obj, Decimal):
            return float(obj)
        elif isinstance(obj, UUID):
            return str(obj)
        return super().default(obj)


def fetch_data_from_postgres(query, db_config):
    """
    Fetches data from PostgreSQL and returns it as a JSON string.
    Native Python types are preserved (no conversion to string).
    """
    conn = None
    result_data = []

    try:
        # Connect to PostgreSQL
        conn = psycopg2.connect(**db_config)
        cur = conn.cursor()

        # Execute query
        print(f"\n🔎 Executing query:\n{query}\n")
        cur.execute(query)
        rows = cur.fetchall()

        # Fetch column names
        col_names = [desc[0] for desc in cur.description]

        # Convert rows to list of dictionaries
        for row in rows:
            result_data.append(dict(zip(col_names, row)))

    except psycopg2.Error as e:
        print(f"❌ Error: {e}")
        return json.dumps({"error": str(e)}, indent=4)

    finally:
        if conn:
            conn.close()

    # Use the custom encoder here ✅
    return json.dumps(result_data, indent=4, cls=PostgresDataEncoder)


# ✅ PostgreSQL connection config
db_config = {
    "host": "localhost",
    "database": "postgres",
    "user": "postgres",
    "password": "root",
    "port": "5432"
}

# 📥 Get the query from user
query = input("\n🛠️ Enter your SQL query: ")

# 📤 Fetch and display the result in JSON
json_result = fetch_data_from_postgres(query, db_config)
print("\n📊 Query Result in JSON format:")
print(json_result)



🔎 Executing query:
WITH LaggedData AS (   SELECT     date,     SUM(pieces_produced_life) AS total_produced,     LAG(SUM(pieces_produced_life), 1, 0) OVER (ORDER BY date) AS previous_total   FROM machine_data   GROUP BY date ) SELECT   date,   total_produced - previous_total AS change_in_pieces FROM LaggedData;


📊 Query Result in JSON format:
[
    {
        "date": "2025-03-21",
        "change_in_pieces": 200460638
    },
    {
        "date": "2025-03-23",
        "change_in_pieces": -68293920
    },
    {
        "date": "2025-03-24",
        "change_in_pieces": -81840224
    }
]


In [4]:
# import pandas as pd
# import numpy as np
# import json

# def infer_dataframe_dtypes(json_result):
#     """
#     Takes JSON result from SQL and returns a pandas DataFrame
#     with inferred and converted column data types.
#     """

#     # Load JSON and convert to DataFrame
#     data = json.loads(json_result)
#     df = pd.DataFrame(data)

#     # Optional: normalize column names
#     df.columns = [col.lower() for col in df.columns]

#     # Define known epoch columns (ms)
#     epoch_cols = ['start_timestamp', 'end_timestamp']

#     # Columns to force datetime parsing from string
#     datetime_cols = ['date', 'local_time']

#     for col in df.columns:
#         series = df[col]

#         # Epoch timestamp conversion
#         if col in epoch_cols and pd.api.types.is_integer_dtype(series):
#             df[col] = pd.to_datetime(series.astype('Int64'), unit='ms', utc=True, errors='coerce')
#             continue

#         # Forced datetime conversion for known datetime-like fields
#         if col in datetime_cols:
#             try:
#                 df[col] = pd.to_datetime(series, utc=True, errors='coerce')
#             except Exception as e:
#                 print(f"⚠️ Failed to parse datetime for '{col}': {e}")
#             continue

#         # General inference from values
#         inferred = pd.api.types.infer_dtype(series, skipna=True)
#         print(f"🔍 Column '{col}' inferred as: {inferred}")
#         if inferred in ['datetime', 'datetime64']:
#             df[col] = pd.to_datetime(series, errors='coerce')

#         elif inferred == 'boolean':
#             df[col] = series.astype('boolean')

#         elif inferred == 'integer':
#             if len(str(df[col][1])) >=10:
#                     df[col] = pd.to_datetime(series, utc=True, errors='coerce')
                
#             else:
#                 df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')

#         elif inferred == 'floating':
#             df[col] = pd.to_numeric(series, errors='coerce', downcast='float')

#         elif inferred in ['string', 'unicode']:
#             # Try datetime
#             try:
#                 df[col] = pd.to_datetime(series, utc=True, errors='coerce')
#                 continue
#             except Exception:
#                 pass
#             unique_ratio = series.nunique(dropna=True) / len(series)
#             if unique_ratio < 0.5:
#                 df[col] = series.astype('category')
#             else:
#                 df[col] = series.astype('string')

#         elif inferred == 'object':
#             # Try datetime
#             try:
#                 df[col] = pd.to_datetime(series, utc=True, errors='coerce')
#                 continue
#             except Exception:
#                 pass

#             # Try numeric
#             try:
#                 df[col] = pd.to_numeric(series, errors='raise')
#                 continue
#             except Exception:
#                 pass

#             # Try category
#             unique_ratio = series.nunique(dropna=True) / len(series)
#             if unique_ratio < 0.5:
#                 df[col] = series.astype('category')
#             else:
#                 df[col] = series.astype('string')

#     # ✅ Final datatypes
#     print("\n📊 Final DataFrame dtypes:")
#     print(df.dtypes)

#     # Optional: return or preview
#     print("\n🔍 Sample Data:")
#     # print(df.head())

#     return df
# df=infer_dataframe_dtypes(json_result)

import pandas as pd
import numpy as np
import json
import logging

# Optional: configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

def try_parse_datetime(series):
    """Try parsing datetime from a series and return parsed version if valid."""
    try:
        parsed = pd.to_datetime(series, utc=True, errors='coerce')
        if parsed.notna().sum() > 0:
            return parsed
    except Exception:
        pass
    return None

def infer_dataframe_dtypes(json_result):
    """
    Converts a JSON result into a pandas DataFrame with smart dtype inference,
    including timestamps, categories, numerics, and datetimes.
    """
    # Load JSON and convert to DataFrame
    data = json.loads(json_result)
    df = pd.DataFrame(data)

    # Normalize column names
    df.columns = [col.lower() for col in df.columns]

    # Define explicit datetime string columns
    datetime_cols = {'date', 'local_time'}

    for col in df.columns:
        series = df[col]

        # Skip empty or all-null columns
        if series.dropna().empty:
            df[col] = series.astype('string')
            continue

        # Explicit datetime parsing
        if col in datetime_cols:
            parsed = try_parse_datetime(series)
            if parsed is not None:
                df[col] = parsed
            else:
                logger.warning(f"⚠️ Failed to parse datetime for '{col}'")
            continue

        # Auto-detect epoch timestamps (>=13 digits)
        if pd.api.types.is_integer_dtype(series):
            max_val = series.dropna().max()
            if max_val and max_val > 1e12:  # Likely epoch in ms
                df[col] = pd.to_datetime(series, unit='ms', utc=True, errors='coerce')
                continue
            else:
                df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')
                continue

        # Infer general dtype
        inferred = pd.api.types.infer_dtype(series, skipna=True)
        logger.info(f"🔍 Column '{col}' inferred as: {inferred}")

        if inferred in ['datetime', 'datetime64']: 
            df[col] = try_parse_datetime(series) or series

        elif inferred == 'boolean':
            df[col] = series.astype('boolean')

        elif inferred == 'integer':
            # Check for epoch-like long integer
            max_len = series.dropna().astype(str).str.len().max()
            if max_len is not None and max_len >= 13:
                df[col] = pd.to_datetime(series, unit='ms', utc=True, errors='coerce')
            else:
                df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')

        elif inferred == 'floating':
            df[col] = pd.to_numeric(series, errors='coerce', downcast='float')

        elif inferred in ['string', 'unicode', 'mixed', 'object']:
            parsed = try_parse_datetime(series)
            if parsed is not None:
                df[col] = parsed
                continue

            # Try numeric
            try:
                df[col] = pd.to_numeric(series, errors='raise')
                continue
            except Exception:
                pass

            # Try categorization
            unique_ratio = series.nunique(dropna=True) / len(series)
            if unique_ratio < 0.5:
                df[col] = series.astype('category')
            else:
                df[col] = series.astype('string')

    # Final DataFrame info
    logger.info("\n📊 Final DataFrame dtypes:")
    logger.info(df.dtypes)

    logger.info("\n🔍 Sample Data:")
    logger.info(df.head())

    return df
df = infer_dataframe_dtypes(json_result)

INFO:__main__:
📊 Final DataFrame dtypes:
INFO:__main__:date                datetime64[ns, UTC]
change_in_pieces                  Int64
dtype: object
INFO:__main__:
🔍 Sample Data:
INFO:__main__:                       date  change_in_pieces
0 2025-03-21 00:00:00+00:00         200460638
1 2025-03-23 00:00:00+00:00         -68293920
2 2025-03-24 00:00:00+00:00         -81840224


In [ ]:
df.dtypes

In [ ]:
df.value_counts()

In [5]:
# import pandas as pd
# import numpy as np
# import json
# from datetime import datetime
# import plotly.express as px




# # ✅ Handle NaN values properly
# # df.replace("NaN", np.nan, inplace=True)

# import pandas as pd
# import numpy as np
# import plotly.express as px
# from datetime import datetime

# def is_distribution_column(series):
#     return (
#         pd.api.types.is_numeric_dtype(series) and
#         series.nunique() > 10 and
#         not series.name.lower().endswith('id') and
#         series.std() > 0
#     )

# def feature_type_and_Plot(df):
#     df.replace("NaN", np.nan, inplace=True)
# #
#     # Optional conversion
#     if 'shiftid' in df.columns:
#         df['shiftid'] = df['shiftid'].astype('category')

#     num_cols = df.select_dtypes(include=['number']).columns.tolist()
#     cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
#     datetime_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()

#     print(f"Numerical columns: {num_cols}")
#     print(f"Categorical columns: {cat_cols}")
#     print(f"Datetime columns: {datetime_cols}")

#     # 1. Time Series Plot
#     if any(col in df.columns for col in datetime_cols):
#         time_col = next((col for col in datetime_cols if col in df.columns), None)
#         if time_col and num_cols:
#             return px.line(df, x=time_col, y=num_cols[0], title=f"Time Trend of {num_cols[0]}")

#     # 2. One numerical column
#     if len(num_cols) == 1:
#         col = num_cols[0]
#         is_dist_col = is_distribution_column(df[col])

#         if len(cat_cols) == 1:
#             cat = cat_cols[0]

#             if is_dist_col:
#                 fig_box = px.box(df, x=cat, y=col, title=f"{col} Distribution by {cat}")
#                 fig_box.show()

#                 fig_violin = px.violin(df, x=cat, y=col, box=True, points="all",
#                                        title=f"Violin Plot of {col} by {cat}")
#                 fig_violin.show()

#                 fig_strip = px.strip(df, x=cat, y=col, title=f"Strip Plot of {col} by {cat}")
#                 fig_strip.show()

#             else:
#                 agg_df = df.groupby(cat)[col].mean().reset_index()
#                 fig_bar = px.bar(agg_df, x=cat, y=col, title=f"Average {col} by {cat}", color=cat)
#                 fig_bar.show()

#             return None

#         else:
#             if is_dist_col:
#                 fig_hist = px.histogram(df, x=col, title=f"Distribution of {col}", nbins=30)
#                 fig_hist.show()

#                 fig_box = px.box(df, y=col, title=f"Box Plot of {col}")
#                 fig_box.show()

#             return None

#     # 3. One categorical + one numerical → Pie or Bar
#     if len(cat_cols) == 1 and len(num_cols) == 1:
#         cat = cat_cols[0]
#         col = num_cols[0]

#         unique_vals = df[cat].nunique()
#         if unique_vals <= 10:
#             return px.pie(df, names=cat, values=col, title=f"Pie Chart of {col} by {cat}")
#         else:
#             return px.bar(df, x=cat, y=col, title=f"{col} by {cat} (Bar Plot)")

#     # 4. Two numerical columns → Scatter
#     if len(num_cols) == 2:
#         return px.scatter(df, x=num_cols[0], y=num_cols[1], title=f"{num_cols[0]} vs {num_cols[1]}")

#     # 5. Multiple numericals (3–4) → Scatter matrix
#     if 3 <= len(num_cols) <= 4:
#         return px.scatter_matrix(df, dimensions=num_cols, title="Scatter Matrix of Numerical Columns")

#     # 6. Only categoricals → Sunburst
#     if len(cat_cols) >= 2 and not num_cols:
#         return px.sunburst(df, path=cat_cols[:3], title="Category Hierarchy (Sunburst)")

#     # 7. One cat + one num again → Strip plot fallback
#     if len(cat_cols) == 1 and len(num_cols) == 1:
#         return px.strip(df, x=cat_cols[0], y=num_cols[0], title=f"Strip Plot of {num_cols[0]} by {cat_cols[0]}")

#     # 8. Heatmap for correlation (if many numerical)
#     if len(num_cols) >= 3:
#         corr_df = df[num_cols].corr()
#         fig = px.imshow(corr_df, text_auto=True, title="Correlation Heatmap")
#         return fig

#     # ✅ Final fallback
#     if num_cols:
#         return px.histogram(df, x=num_cols[0], title=f"Fallback Histogram for {num_cols[0]}")
#     elif cat_cols:
#         return px.bar(df[cat_cols[0]].value_counts().reset_index(), 
#                       x='index', y=cat_cols[0],
#                       title=f"Fallback Count Plot for {cat_cols[0]}")
#     else:
#         print("No suitable columns to plot.")
#         return None

# # === Example usage ===

# fig = feature_type_and_Plot(df)
# if fig: fig.show()


import pandas as pd
import numpy as np
import plotly.express as px

def feature_type_and_Plot(df):
    df.replace("NaN", np.nan, inplace=True)

    if 'shiftid' in df.columns:
        df['shiftid'] = df['shiftid'].astype('category')

    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    datetime_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()

    print(f"Numerical columns: {num_cols}")
    print(f"Categorical columns: {cat_cols}")
    print(f"Datetime columns: {datetime_cols}")

    distribution_cols = [col for col in num_cols if df[col].nunique() > 10]

    # ✅ CASE: Multi-line Time Series - Multiple numerical columns
    if len(datetime_cols) >= 1 and len(distribution_cols) >= 2:
        time_col = datetime_cols[0]
        return px.line(df, x=time_col, y=distribution_cols[:2], 
                       title=f"Multiple Metrics Over Time: {', '.join(distribution_cols[:2])}")

    # ✅ CASE: Multi-line Time Series - One numerical + one categorical over time
    if len(datetime_cols) == 1 and len(distribution_cols) == 1 and len(cat_cols) == 1:
        time_col = datetime_cols[0]
        y_col = distribution_cols[0]
        color_col = cat_cols[0]
        return px.line(df, x=time_col, y=y_col, color=color_col,
                       title=f"{y_col} Over Time by {color_col}")

    # ✅ CASE: Single time series (fallback)
    if datetime_cols and distribution_cols:
        time_col = datetime_cols[0]
        return px.line(df, x=time_col, y=distribution_cols[0], 
                       title=f"Time Trend of {distribution_cols[0]}")

    # ✅ CASE: One numerical + one categorical
    if len(distribution_cols) == 1 and len(cat_cols) == 1:
        col = distribution_cols[0]
        cat = cat_cols[0]

        # fig_bar = px.bar(df, x=cat, y=col, title=f"Average {col} by {cat}", color=cat)
        # fig_bar.show()

        fig_box = px.box(df, x=cat, y=col, title=f"{col} Distribution by {cat}")
        fig_box.show()

        fig_violin = px.violin(df, x=cat, y=col, box=True, points="all", 
                               title=f"Violin Plot of {col} by {cat}")
        fig_violin.show()
        return None
    
    elif len(num_cols) == 1 and len(cat_cols) == 1:
        fig_bar = px.bar(df, x=cat_cols[0], y=num_cols[0], title=f"Bar Plot for {num_cols[0]} by {cat_cols[0]}", color=cat_cols[0])
        fig_bar.show()
        
    # ✅ CASE: Only numerical
    if len(distribution_cols) == 1:
        col = distribution_cols[0]
        fig_hist = px.histogram(df, x=col, title=f"Distribution of {col}", nbins=30)
        fig_hist.show()

        fig_box = px.box(df, y=col, title=f"Box Plot of {col}")
        fig_box.show()
        return None

    # ✅ CASE: Pie Chart for low cardinality category
    if len(cat_cols) == 1 and len(distribution_cols) == 1:
        cat = cat_cols[0]
        col = distribution_cols[0]

        if df[cat].nunique() <= 10:
            return px.pie(df, names=cat, values=col, title=f"Pie Chart of {col} by {cat}")
        else:
            return px.bar(df, x=cat, y=col, title=f"{col} by {cat} (Bar Plot)")

    # ✅ CASE: two numericals
    sequential_num_cols = [
    col for col in num_cols
    if df[col].is_monotonic_increasing and df[col].nunique() > 1 ]
    
    print(f"Sequential numerical columns: {sequential_num_cols}")

    if len(num_cols) == 2:
        
        if num_cols[0] in sequential_num_cols:
            return px.line(df, x=num_cols[0], y=num_cols[1], 
                           title=f"Line Plot of {num_cols[0]} vs {num_cols[1]}")
            
        elif num_cols[1] in sequential_num_cols:
            return px.line(df, x=num_cols[1], y=num_cols[0], 
                           title=f"Line Plot of {num_cols[1]} vs {num_cols[0]}")
        
        else:
            return px.scatter(df, x=num_cols[0], y=num_cols[1], 
                              title=f"Scatter Plot of {num_cols[0]} vs {num_cols[1]}")
        

    # ✅ CASE: Scatter matrix
    if 3 <= len(distribution_cols) <= 3:
        return px.scatter_matrix(df, dimensions=distribution_cols, title="Scatter Matrix")

    # ✅ CASE: Only categoricals
    if len(cat_cols) >= 2 and not distribution_cols:
        return px.sunburst(df, path=cat_cols[:3], title="Category Hierarchy (Sunburst)")

    # ✅ CASE: Strip plot
    if len(cat_cols) == 1 and len(distribution_cols) == 1:
        return px.strip(df, x=cat_cols[0], y=distribution_cols[0], 
                        title=f"Strip Plot of {distribution_cols[0]} by {cat_cols[0]}")

    # ✅ CASE: Correlation heatmap
    if len(distribution_cols) > 3:
        corr_df = df[distribution_cols].corr()
        return px.imshow(corr_df, text_auto=True, title="Correlation Heatmap")

    # ✅ Fallback plot
    if distribution_cols:
        return px.box(df, y=distribution_cols[0], title=f"Fallback: Box Plot of {distribution_cols[0]}")

    return None

fig = feature_type_and_Plot(df)
if fig: 
    fig.show()


Numerical columns: ['change_in_pieces']
Categorical columns: []
Datetime columns: ['date']
Sequential numerical columns: []


In [2]:
import pandas as pd
import numpy as np
import json
import logging
from datetime import datetime
import plotly.express as px

# Optional: configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Helper function to try parsing a series as datetime
def try_parse_datetime(series):
    try:
        # Attempt to parse the series as datetime
        parsed = pd.to_datetime(series, utc=True, errors='coerce')
        if parsed.notna().sum() > 0:
            return parsed
    except Exception:
        pass
    return None

# Function to infer data types of a DataFrame from a JSON string
def infer_dataframe_dtypes(json_result):
    # Load JSON string into a dictionary
    data = json.loads(json_result)
    # Convert dictionary to DataFrame
    df = pd.DataFrame(data)
    # Convert column names to lowercase
    df.columns = [col.lower() for col in df.columns]
    # Define columns that are likely to be datetime
    datetime_cols = {'date', 'local_time'}

    # Iterate through each column to infer its data type
    for col in df.columns:
        series = df[col]

        # Handle empty columns
        if series.dropna().empty:
            df[col] = series.astype('string')
            continue

        # Handle datetime columns
        if col in datetime_cols:
            parsed = try_parse_datetime(series)
            if parsed is not None:
                df[col] = parsed
            else:
                logger.warning(f"⚠️ Failed to parse datetime for '{col}'")
            continue

        # Handle integer columns
        if pd.api.types.is_integer_dtype(series):
            max_val = series.dropna().max()
            if max_val and max_val > 1e12:  # Likely a timestamp in milliseconds
                df[col] = pd.to_datetime(series, unit='ms', utc=True, errors='coerce')
                continue
            else:
                df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')
                continue

        # Infer data type using pandas' infer_dtype
        inferred = pd.api.types.infer_dtype(series, skipna=True)
        logger.info(f"🔍 Column '{col}' inferred as: {inferred}")

        # Handle inferred data types
        if inferred in ['datetime', 'datetime64']:
            df[col] = try_parse_datetime(series) or series
        elif inferred == 'boolean':
            df[col] = series.astype('boolean')
        elif inferred == 'integer':
            max_len = series.dropna().astype(str).str.len().max()
            if max_len is not None and max_len >= 13:  # Likely a timestamp in milliseconds
                df[col] = pd.to_datetime(series, unit='ms', utc=True, errors='coerce')
            else:
                df[col] = pd.to_numeric(series, errors='coerce').astype('Int64')
        elif inferred == 'floating':
            df[col] = pd.to_numeric(series, errors='coerce', downcast='float')
        elif inferred in ['string', 'unicode', 'mixed', 'object']:
            parsed = try_parse_datetime(series)
            if parsed is not None:
                df[col] = parsed
                continue
            try:
                df[col] = pd.to_numeric(series, errors='raise')
                continue
            except Exception:
                pass
            unique_ratio = series.nunique(dropna=True) / len(series)
            if unique_ratio < 0.5:  # If unique values are less than 50%, treat as category
                df[col] = series.astype('category')
            else:
                df[col] = series.astype('string')

    # Log final DataFrame dtypes and sample data
    logger.info("\n📊 Final DataFrame dtypes:")
    logger.info(df.dtypes)

    logger.info("\n🔍 Sample Data:")
    logger.info(df.head())

    return df

# Function to determine feature types and generate plots
def feature_type_and_Plot(df):
    df.replace("NaN", np.nan, inplace=True)

    if 'shiftid' in df.columns:
        df['shiftid'] = df['shiftid'].astype('category')

    num_cols = df.select_dtypes(include=['number']).columns.tolist()
    cat_cols = df.select_dtypes(include=['object', 'category', 'string']).columns.tolist()
    datetime_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()

    print(f"Numerical columns: {num_cols}")
    print(f"Categorical columns: {cat_cols}")
    print(f"Datetime columns: {datetime_cols}")

    distribution_cols = [col for col in num_cols if df[col].nunique() > 10]

    # ✅ CASE: Multi-line Time Series - Multiple numerical columns
    if len(datetime_cols) >= 1 and len(distribution_cols) >= 2:
        time_col = datetime_cols[0]
        return px.line(df, x=time_col, y=distribution_cols[:2], 
                       title=f"Multiple Metrics Over Time: {', '.join(distribution_cols[:2])}")

    # ✅ CASE: Multi-line Time Series - One numerical + one categorical over time
    if len(datetime_cols) == 1 and len(num_cols) == 1 and len(cat_cols) == 1:
        time_col = datetime_cols[0]
        y_col = num_cols[0]
        color_col = cat_cols[0]
        return px.line(df, x=time_col, y=y_col, color=color_col,
                       title=f"{y_col} Over Time by {color_col}")

    # ✅ CASE: Single time series (fallback)
    if datetime_cols and distribution_cols:
        time_col = datetime_cols[0]
        return px.line(df, x=time_col, y=distribution_cols[0], 
                       title=f"Time Trend of {distribution_cols[0]}")

    # ✅ CASE: One numerical + one categorical
    if len(distribution_cols) == 1 and len(cat_cols) == 1:
        col = distribution_cols[0]
        cat = cat_cols[0]

        # fig_bar = px.bar(df, x=cat, y=col, title=f"Average {col} by {cat}", color=cat)
        # fig_bar.show()

        fig_box = px.box(df, x=cat, y=col, title=f"{col} Distribution by {cat}")
        fig_box.show()

        fig_violin = px.violin(df, x=cat, y=col, box=True, points="all", 
                               title=f"Violin Plot of {col} by {cat}")
        fig_violin.show()
        return None
    
    elif len(num_cols) == 1 and len(cat_cols) == 1:
        fig_bar = px.bar(df, x=cat_cols[0], y=num_cols[0], title=f"Bar Plot for {num_cols[0]} by {cat_cols[0]}", color=cat_cols[0])
        fig_bar.show()
        
    # ✅ CASE: Only numerical
    if len(distribution_cols) == 1:
        col = distribution_cols[0]
        fig_hist = px.histogram(df, x=col, title=f"Distribution of {col}", nbins=30)
        fig_hist.show()

        fig_box = px.box(df, y=col, title=f"Box Plot of {col}")
        fig_box.show()
        return None

    # ✅ CASE: Pie Chart for low cardinality category
    if len(cat_cols) == 1 and len(distribution_cols) == 1:
        cat = cat_cols[0]
        col = distribution_cols[0]

        if df[cat].nunique() <= 10:
            return px.pie(df, names=cat, values=col, title=f"Pie Chart of {col} by {cat}")
        else:
            return px.bar(df, x=cat, y=col, title=f"{col} by {cat} (Bar Plot)")

    # ✅ CASE: two numericals
    sequential_num_cols = [
    col for col in num_cols
    if df[col].is_monotonic_increasing and df[col].nunique() > 1 ]
    
    print(f"Sequential numerical columns: {sequential_num_cols}")

    if len(num_cols) == 2:
        
        if num_cols[0] in sequential_num_cols:
            return px.line(df, x=num_cols[0], y=num_cols[1], 
                           title=f"Line Plot of {num_cols[0]} vs {num_cols[1]}")
            
        elif num_cols[1] in sequential_num_cols:
            return px.line(df, x=num_cols[1], y=num_cols[0], 
                           title=f"Line Plot of {num_cols[1]} vs {num_cols[0]}")
        
        else:
            return px.scatter(df, x=num_cols[0], y=num_cols[1], 
                              title=f"Scatter Plot of {num_cols[0]} vs {num_cols[1]}")
        

    # ✅ CASE: Scatter matrix
    if 3 <= len(distribution_cols) <= 3:
        return px.scatter_matrix(df, dimensions=distribution_cols, title="Scatter Matrix")

    # ✅ CASE: Only categoricals
    if len(cat_cols) >= 2 and not distribution_cols:
        return px.sunburst(df, path=cat_cols[:3], title="Category Hierarchy (Sunburst)")

    # ✅ CASE: Strip plot
    if len(cat_cols) == 1 and len(distribution_cols) == 1:
        return px.strip(df, x=cat_cols[0], y=distribution_cols[0], 
                        title=f"Strip Plot of {distribution_cols[0]} by {cat_cols[0]}")

    # ✅ CASE: Correlation heatmap
    if len(distribution_cols) > 3:
        corr_df = df[distribution_cols].corr()
        return px.imshow(corr_df, text_auto=True, title="Correlation Heatmap")

    # ✅ Fallback plot
    if distribution_cols:
        return px.box(df, y=distribution_cols[0], title=f"Fallback: Box Plot of {distribution_cols[0]}")

    return None

# Example runner
if __name__ == "__main__":
    
    df = infer_dataframe_dtypes(json_result)
    fig = feature_type_and_Plot(df)
    if fig:
        fig.show()

    print("\n✅ Final dtypes:")
    print(df.dtypes)
    print("\n🧾 Data Sample:")
    print(df.head())


INFO:__main__:
📊 Final DataFrame dtypes:
INFO:__main__:date                datetime64[ns, UTC]
change_in_pieces                  Int64
dtype: object
INFO:__main__:
🔍 Sample Data:
INFO:__main__:                       date  change_in_pieces
0 2025-03-21 00:00:00+00:00         200460638
1 2025-03-23 00:00:00+00:00         -68293920
2 2025-03-24 00:00:00+00:00         -81840224


Numerical columns: ['change_in_pieces']
Categorical columns: []
Datetime columns: ['date']
Sequential numerical columns: []

✅ Final dtypes:
date                datetime64[ns, UTC]
change_in_pieces                  Int64
dtype: object

🧾 Data Sample:
                       date  change_in_pieces
0 2025-03-21 00:00:00+00:00         200460638
1 2025-03-23 00:00:00+00:00         -68293920
2 2025-03-24 00:00:00+00:00         -81840224
